# Icelandic ASR test evaluation · Person 4

Evaluate saved predictions on the frozen 1,884-clip Samrómur 21.05 test subset. Run the cells in order on a standard CPU runtime. The notebook fetches lightweight inputs at a pinned project commit and writes the tables, methods, and figure under `person4_evaluation` in the working directory (normally `/content` in Colab). Download that folder's files before ending the session, or set `results_dir` to a mounted Drive path.

This version covers Person 2's four models. The planned comparison of the team's fine-tuned Whisper against the original Whisper Small is pending final test predictions. Use the configuration below when a compatible CSV is available. The published Icelandic wav2vec 2.0 and Whisper Small checkpoints may have seen some of these test recordings during external training.


In [ ]:
%pip install -q jiwer==4.0.0 soundfile

## 1. Load the frozen references and saved predictions

To add a model, configure its label and CSV path in `extra_predictions`. The expected columns are `id`, `split`, `speaker_session_id`, `reference`, `reference_raw`, and `hypothesis_raw`. Preserve clip IDs as strings, including leading zeros, and keep rows with empty hypotheses. A differently structured CSV needs a column mapping before adding it here.


In [ ]:
import csv
import json
import runpy
import subprocess
import sys
from collections import Counter
from pathlib import Path

source_commit = "e9cd03331ef7f2295a5baf197c8c74f1ce41417d"
repo_url = "https://github.com/RanimAlyaseen/icelandic-asr-project3.git"
inputs = Path.cwd() / "person4_inputs"
results_dir = Path.cwd() / "person4_evaluation"
extra_predictions = {}  # Example: {"Team fine-tuned Whisper": Path("/content/verified_test.csv")}

pretrained = "results/pretrained/predictions"
model_sources = {
    "hubert_ctc_test.csv": "results/hubert_ctc/predictions/hubert_ctc_test.csv",
    "wav2vec2_is_test.csv": f"{pretrained}/wav2vec2_is_test.csv",
    "whisper_small_is_test_beam5.csv": f"{pretrained}/whisper_small_is_test_beam5.csv",
    "whisper_small_test_beam5.csv": f"{pretrained}/whisper_small_test_beam5.csv",
}
inputs.mkdir(exist_ok=True)
git_dir = inputs / "repository"
git_dir.mkdir(exist_ok=True)

def git(*args, **kwargs):
    return subprocess.run(["git", "-C", str(git_dir), *args], check=True, **kwargs)

git("init", "-q")
git("fetch", "-q", "--depth=1", "--no-tags", repo_url, source_commit)

source_files = {
    "evaluation_references.tsv": "output/evaluation_references.tsv",
    "test.jsonl": "output/test.jsonl",
    "prepare.py": "prepare.py",
    "audit.py": "audit.py",
    **model_sources,
}
for filename, repo_path in source_files.items():
    blob = git("show", f"{source_commit}:{repo_path}", capture_output=True).stdout
    (inputs / filename).write_bytes(blob)

model_paths = {name: inputs / name for name in model_sources}
if model_paths.keys() & extra_predictions.keys():
    raise ValueError("An added model duplicates a baseline name.")
model_paths.update({label: Path(path) for label, path in extra_predictions.items()})
model_files = list(model_paths)

# Use the same prepare.py and its local dependency from the pinned commit.
sys.path.insert(0, str(inputs))
try:
    normalize_text = runpy.run_path(str(inputs / "prepare.py"))["normalize_text"]
finally:
    sys.path.pop(0)

def read_rows(path, delimiter=","):
    with path.open(encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file, delimiter=delimiter))

print(f"Source commit: {source_commit}")

## 2. Verify clip IDs, references, and session groups

In [ ]:
references = [
    row for row in read_rows(inputs / "evaluation_references.tsv", "\t")
    if row["split"] == "test"
]
official = {row["id"]: row for row in references}
assert len(references) == len(official) == 1884, "Check the frozen test references."

with (inputs / "test.jsonl").open(encoding="utf-8") as file:
    manifest = [json.loads(line) for line in file if line.strip()]
manifest_by_id = {row["id"]: row for row in manifest}
assert len(manifest) == len(manifest_by_id) == len(official)
assert manifest_by_id.keys() == official.keys(), "Test clip IDs differ."
assert all(row["split"] == "test" and row["speaker_session_id"] for row in manifest)
session_by_clip = {
    clip_id: str(row["speaker_session_id"]) for clip_id, row in manifest_by_id.items()
}
assert len(set(session_by_clip.values())) == 1026, "Unexpected session count."

def validate_predictions(rows, frozen, sessions):
    counts = Counter(row["id"] for row in rows)
    return {
        "duplicate IDs": sum(count > 1 for count in counts.values()),
        "missing IDs": len(frozen.keys() - counts.keys()),
        "unexpected IDs": len(counts.keys() - frozen.keys()),
        "wrong split": sum(row["split"] != "test" for row in rows),
        "reference mismatches": sum(
            row["id"] in frozen and (
                row["reference"] != frozen[row["id"]]["text"]
                or row["reference_raw"] != frozen[row["id"]]["text_raw"]
            )
            for row in rows
        ),
        "session mismatches": sum(
            row["id"] in sessions and row["speaker_session_id"] != sessions[row["id"]]
            for row in rows
        ),
        "malformed hypothesis fields": sum(row["hypothesis_raw"] is None for row in rows),
    }

# Verify exact matching, preservation of empty predictions, and duplicate detection.
example_ref = {"0001": {"text": "ég", "text_raw": "Ég."}}
example_sessions = {"0001": "s1"}
example = {
    "id": "0001", "split": "test", "speaker_session_id": "s1",
    "reference": "ég", "reference_raw": "Ég.", "hypothesis_raw": "",
}
assert not any(validate_predictions([example], example_ref, example_sessions).values())
assert validate_predictions([example, example], example_ref, example_sessions)["duplicate IDs"] == 1
bad_example = {**example, "reference": "þú", "speaker_session_id": "s2"}
issues = validate_predictions([bad_example], example_ref, example_sessions)
assert issues["reference mismatches"] == issues["session mismatches"] == 1

rows_by_model = {name: read_rows(path) for name, path in model_paths.items()}
for name, rows in rows_by_model.items():
    issues = validate_predictions(rows, official, session_by_clip)
    failures = {issue: count for issue, count in issues.items() if count}
    if failures:
        raise ValueError(f"{name}: {failures}")
    print(f"PASS {name} | {len(rows):,} rows")
print(f"Verified {len(set(session_by_clip.values())):,} speaker/session groups.")

## 3. Recalculate WER and CER

Calculate corpus-level scores against the frozen references with shared text normalization. Empty hypotheses count as deletions when the reference has words.


In [ ]:
import jiwer

reported = {
    "hubert_ctc_test.csv": (68.46, 23.31),
    "wav2vec2_is_test.csv": (7.76, 1.86),
    "whisper_small_is_test_beam5.csv": (21.75, 5.60),
    "whisper_small_test_beam5.csv": (86.82, 36.35),
}

for filename, rows in rows_by_model.items():
    expected = reported.get(filename)
    refs = [normalize_text(official[row["id"]]["text"]) for row in rows]
    preds = [normalize_text(row["hypothesis_raw"]) for row in rows]

    wer = round(100 * jiwer.wer(refs, preds), 2)
    cer = round(100 * jiwer.cer(refs, preds), 2)
    if expected is not None and (wer, cer) != expected:
        raise ValueError(f"{filename}: recalculated {(wer, cer)} vs reported {expected}")
    status = "MATCH" if expected is not None else "SCORE"

    print(f"{status:5}  {filename:34} WER {wer:6.2f}%  CER {cer:6.2f}%")

## 4. Calculate 95% confidence intervals

Resample whole speaker/session groups 10,000 times with replacement using seed 2105. Each draw uses the same groups for all models. Recompute corpus rates from the sum of errors divided by the sum of reference units. The interval endpoints are the 2.5th and 97.5th percentiles.


In [ ]:
import numpy as np

sessions = sorted(set(session_by_clip.values()))
session_index = {session: i for i, session in enumerate(sessions)}
normalized_refs = {
    clip_id: normalize_text(row["text"]) for clip_id, row in official.items()
}

# First two columns are reference word and character counts.
# Each model then gets two columns for word and character errors.
session_totals = np.zeros((len(sessions), 2 + 2 * len(model_files)), dtype=np.int64)

for clip_id, reference in normalized_refs.items():
    i = session_index[session_by_clip[clip_id]]
    session_totals[i, :2] += [len(reference.split()), len(reference)]

for model_index, filename in enumerate(model_files):
    column = 2 + 2 * model_index
    for row in rows_by_model[filename]:
        clip_id = row["id"]
        i = session_index[session_by_clip[clip_id]]
        reference = normalized_refs[clip_id]
        prediction = normalize_text(row["hypothesis_raw"])

        words = jiwer.process_words(reference, prediction)
        chars = jiwer.process_characters(reference, prediction)
        session_totals[i, column:column + 2] += [
            words.substitutions + words.deletions + words.insertions,
            chars.substitutions + chars.deletions + chars.insertions,
        ]

seed = 2105
n_resamples = 10_000
rng = np.random.default_rng(seed)
bootstrap_totals = np.empty((n_resamples, session_totals.shape[1]), dtype=np.int64)

for start in range(0, n_resamples, 250):
    stop = min(start + 250, n_resamples)
    drawn = rng.integers(0, len(sessions), size=(stop - start, len(sessions)))
    bootstrap_totals[start:stop] = session_totals[drawn].sum(axis=1)

observed_totals = session_totals.sum(axis=0)
ci_rows = []

for model_index, filename in enumerate(model_files):
    for offset, metric in enumerate(("WER", "CER")):
        column = 2 + 2 * model_index + offset
        point = 100 * observed_totals[column] / observed_totals[offset]
        bootstrap_rates = 100 * bootstrap_totals[:, column] / bootstrap_totals[:, offset]
        low, high = np.percentile(bootstrap_rates, [2.5, 97.5])

        if filename in reported:
            assert round(point, 2) == reported[filename][offset]
        ci_rows.append({
            "model_file": filename,
            "metric": metric,
            "score_percent": point,
            "ci_low_percent": low,
            "ci_high_percent": high,
        })
        print(f"{filename:34} {metric} {point:6.2f}%  [{low:.2f}%, {high:.2f}%]")

## 5. Compare models in pairs

Differences are model A minus model B in percentage points. The paired percentile intervals reuse the same bootstrap draws and are unadjusted. For p-values, the two-sided paired randomization test swaps model labels by session under an exchangeability assumption. Use 10,000 random draws (seed 2106), a plus-one correction, and Holm adjustment across the displayed tests. These intervals describe test-set sampling variation.


In [ ]:
from itertools import combinations

comparisons = []
for a, b in combinations(range(len(model_files)), 2):
    for offset, metric in enumerate(("WER", "CER")):
        comparisons.append((a, b, offset, metric))

# Positive differences mean model A has the higher error rate.
group_differences = np.column_stack([
    session_totals[:, 2 + 2 * a + offset]
    - session_totals[:, 2 + 2 * b + offset]
    for a, b, offset, metric in comparisons
])
observed_differences = group_differences.sum(axis=0)

permutation_seed = 2106
n_permutations = 10_000
rng = np.random.default_rng(permutation_seed)
extreme_counts = np.zeros(len(comparisons), dtype=np.int64)

for start in range(0, n_permutations, 250):
    batch_size = min(250, n_permutations - start)
    signs = 2 * rng.integers(0, 2, size=(batch_size, len(sessions))) - 1
    permuted = signs @ group_differences
    extreme_counts += (
        np.abs(permuted) >= np.abs(observed_differences)
    ).sum(axis=0)

p_values = (extreme_counts + 1) / (n_permutations + 1)

# Holm adjustment across all WER and CER tests in this run.
order = np.argsort(p_values)
holm_p = np.empty_like(p_values)
holm_p[order] = np.minimum(
    1, np.maximum.accumulate(p_values[order] * np.arange(len(order), 0, -1))
)

paired_rows = []
for i, (a, b, offset, metric) in enumerate(comparisons):
    a_column = 2 + 2 * a + offset
    b_column = 2 + 2 * b + offset
    difference = 100 * observed_differences[i] / observed_totals[offset]
    bootstrap_differences = 100 * (
        bootstrap_totals[:, a_column] - bootstrap_totals[:, b_column]
    ) / bootstrap_totals[:, offset]
    low, high = np.percentile(bootstrap_differences, [2.5, 97.5])

    paired_rows.append({
        "model_a": model_files[a],
        "model_b": model_files[b],
        "metric": metric,
        "difference_pp": difference,
        "ci_low_pp": low,
        "ci_high_pp": high,
        "p_value": p_values[i],
        "p_holm": holm_p[i],
    })

    print(
        f"{Path(model_files[a]).stem} vs {Path(model_files[b]).stem} | "
        f"{metric}: {difference:+.2f} pp [{low:+.2f}, {high:+.2f}] | "
        f"p={p_values[i]:.5f}, Holm p={holm_p[i]:.5f}"
    )

## 6. Summarize recognition errors

An exact word-boundary example preserves the character sequence after removing spaces. It is already represented in the WER alignment counts. Displayed examples are the first occurrence of each category per model in file order.


In [ ]:
from itertools import groupby

error_types = {
    "substitute": "substitution",
    "delete": "deletion",
    "insert": "insertion",
}

error_summary = []
error_examples = []

for filename in model_files:
    totals = Counter()
    examples = {}

    for row in rows_by_model[filename]:
        clip_id = row["id"]
        reference = normalized_refs[clip_id]
        prediction = normalize_text(row["hypothesis_raw"])
        aligned = jiwer.process_words(reference, prediction)

        totals["substitutions"] += aligned.substitutions
        totals["deletions"] += aligned.deletions
        totals["insertions"] += aligned.insertions

        ref_words = aligned.references[0]
        pred_words = aligned.hypotheses[0]
        chunks = aligned.alignments[0]

        def record_example(category, ref_tokens, pred_tokens):
            examples.setdefault(category, {
                "model_file": filename,
                "clip_id": clip_id,
                "category": category,
                "reference_span": " ".join(ref_tokens),
                "prediction_span": " ".join(pred_tokens),
                "reference": reference,
                "prediction": prediction,
            })

        for chunk in chunks:
            if chunk.type in error_types:
                record_example(
                    error_types[chunk.type],
                    ref_words[chunk.ref_start_idx:chunk.ref_end_idx],
                    pred_words[chunk.hyp_start_idx:chunk.hyp_end_idx],
                )

        for is_equal, block in groupby(chunks, key=lambda chunk: chunk.type == "equal"):
            if is_equal:
                continue
            edits = list(block)
            ref_span = ref_words[edits[0].ref_start_idx:edits[-1].ref_end_idx]
            pred_span = pred_words[edits[0].hyp_start_idx:edits[-1].hyp_end_idx]

            if (ref_span and pred_span
                    and len(ref_span) != len(pred_span)
                    and "".join(ref_span) == "".join(pred_span)):
                totals["exact_word_boundary_cases"] += 1
                record_example("word boundary", ref_span, pred_span)

    error_summary.append({
        "model_file": filename,
        **{key: totals[key] for key in (
            "substitutions", "deletions", "insertions", "exact_word_boundary_cases"
        )},
    })
    error_examples.extend(examples.values())

    print(f"\n{filename}")
    print(
        f"  Substitutions: {totals['substitutions']:,} | "
        f"Deletions: {totals['deletions']:,} | "
        f"Insertions: {totals['insertions']:,} | "
        f"Exact word-boundary cases: {totals['exact_word_boundary_cases']:,}"
    )
    for category, example in examples.items():
        ref = example["reference_span"] or "∅"
        pred = example["prediction_span"] or "∅"
        print(f"  {example['clip_id']} | {category}: {ref!r} → {pred!r}")

## 7. Export results and draw the comparison chart

In [ ]:
import pandas as pd

results_dir.mkdir(parents=True, exist_ok=True)

tables = {
    "model_scores_and_confidence_intervals.csv": ci_rows,
    "paired_comparisons.csv": paired_rows,
    "error_counts.csv": error_summary,
    "error_examples.csv": error_examples,
}

for filename, rows in tables.items():
    pd.DataFrame(rows).to_csv(results_dir / filename, index=False)
    print(f"Saved {filename} ({len(rows)} rows)")

methods = {
    "source_commit": source_commit,
    "split": "test",
    "test_clips": len(official),
    "models": model_files,
    "speaker_session_groups": len(sessions),
    "normalization": "prepare.normalize_text from the source commit",
    "confidence_intervals": {
        "method": "95% percentile bootstrap resampling whole speaker/session groups",
        "seed": seed,
        "resamples": n_resamples,
    },
    "paired_tests": {
        "method": "Two-sided paired randomization with model labels swapped by speaker/session group",
        "seed": permutation_seed,
        "random_swaps": n_permutations,
        "p_value_correction": f"Plus-one Monte Carlo correction, then Holm across {len(paired_rows)} comparisons",
        "difference_direction": "Model A minus model B, in percentage points",
    },
    "caveat": (
        "The published Icelandic wav2vec 2.0 and Whisper Small checkpoints "
        "may have seen some Samrómur test clips during external training."
    ),
}

(results_dir / "methods.json").write_text(
    json.dumps(methods, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(f"Saved methods.json in {results_dir}")

In [ ]:
import matplotlib.pyplot as plt

labels = {
    "hubert_ctc_test.csv": "HuBERT + CTC",
    "wav2vec2_is_test.csv": "Icelandic wav2vec 2.0*",
    "whisper_small_is_test_beam5.csv": "Icelandic Whisper Small*",
    "whisper_small_test_beam5.csv": "Original Whisper Small",
}

scores = pd.DataFrame(ci_rows)
model_order = (
    scores[scores["metric"] == "WER"]
    .sort_values("score_percent")["model_file"]
    .tolist()
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
positions = list(range(len(model_order)))

for ax, metric in zip(axes, ("WER", "CER")):
    subset = scores[scores["metric"] == metric].set_index("model_file").loc[model_order]
    values = subset["score_percent"].to_numpy()
    errors = [
        values - subset["ci_low_percent"].to_numpy(),
        subset["ci_high_percent"].to_numpy() - values,
    ]

    ax.errorbar(values, positions, xerr=errors, fmt="o", capsize=4, color="#145A72")
    ax.set_title(metric)
    ax.set_xlabel("Error rate (%)")
    ax.set_xlim(left=0)
    ax.grid(axis="x", alpha=0.25)

axes[0].set_yticks(positions, labels=[labels.get(name, name) for name in model_order])
axes[0].invert_yaxis()
fig.suptitle("Icelandic ASR test results with 95% confidence intervals")
fig.text(0.02, 0.03, "* Published checkpoints may have seen some Samrómur test clips.", fontsize=9)
fig.tight_layout(rect=(0, 0.10, 1, 0.92))

chart_path = results_dir / "test_error_rates.png"
fig.savefig(chart_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {chart_path}")